# GRID Gemma 4 — Train All 4 Tasks (Single Load)

Trains all 4 micro-models in one session without reloading.
Saves LoRA adapters to HuggingFace. GGUF conversion happens on the GRID server.

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps trl==0.22.2

In [2]:
!rm -rf /content/grid
import os, sys, gc, torch

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not os.path.exists('/content/grid'):
    !git clone https://github.com/3pacs/GRID.git /content/grid
    %cd /content/grid
else:
    %cd /content/grid
    !git pull

sys.path.insert(0, '/content/grid')

# Load HF token from .env if available
if not HF_TOKEN:
    try:
        with open('/content/grid/.env') as f:
            for line in f:
                if line.startswith('HF_API_KEY='):
                    HF_TOKEN = line.strip().split('=', 1)[1]
                    os.environ['HF_TOKEN'] = HF_TOKEN
                    break
    except FileNotFoundError:
        pass

print(f'HF Token: {"configured" if HF_TOKEN else "NOT SET"}')

Cloning into '/content/grid'...
remote: Enumerating objects: 11786, done.
remote: Counting objects: 100% (501/501), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 11786 (delta 279), reused 385 (delta 223), pack-reused 11285 (from 1)
Receiving objects: 100% (11786/11786), 95.25 MiB | 14.66 MiB/s, done.
Resolving deltas: 100% (5581/5581), done.
Updating files: 100% (1845/1845), done.
/content/grid
HF Token: NOT SET


In [3]:
# === CONFIG ===
BASE_MODEL = "unsloth/gemma-4-E2B-it"
TASKS = ["signal_classifier", "anomaly_narrator", "edgar_extractor", "knowledge_mapper"]
HF_USERNAME = "stepdadfinance"
MAX_SEQ_LENGTH = 2048
LORA_R = 8
LORA_ALPHA = 8
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10

In [4]:
# Load model ONCE
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Base model loaded: {trainable:,} / {total:,} params")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base model loaded: 0 / 4,173,561,376 params


In [8]:
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.5 MB/s eta 0:00:00


In [9]:
# Train all 4 tasks sequentially, save LoRA adapters
from unsloth.chat_templates import standardize_data_formats, train_on_responses_only
from gemma.training.config import TaskType, TASK_SYSTEM_PROMPTS
from gemma.training.datasets import load_dataset_for_training
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
import torch

test_prompts = {
    "signal_classifier": "Breaking: Federal Reserve announced emergency 50bp rate cut. Treasury yields dropping sharply.",
    "anomaly_narrator": "Feature: SPX_DAILY_RETURN\nValue: -6.8%\nExpected: -0.1%\nZ-score: -5.2\nPeriod: 2026-04-05",
    "edgar_extractor": "Extract: company_name, filing_type, total_revenue\n\nAMAZON.COM INC\nFORM 10-Q\nNet revenue: $155.7 billion",
    "knowledge_mapper": "BlackRock increased Bitcoin ETF holdings to $45B while lobbying SEC for spot Ethereum ETF approval.",
}

results = {}

for task_name in TASKS:
    print(f"\n{'='*60}")
    print(f"  TASK: {task_name}")
    print(f"{'='*60}\n")

    # Attach fresh LoRA
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        random_state=3407,
    )

    # Load dataset
    task = TaskType(task_name)
    dataset = load_dataset_for_training(task)
    dataset = standardize_data_formats(dataset)

    def format_conversations(examples):
        convos = examples["conversations"]
        texts = [
            tokenizer.apply_chat_template(
                convo, tokenize=False, add_generation_prompt=False
            ).removeprefix("<bos>")
            for convo in convos
        ]
        return {"text": texts}

    dataset = dataset.map(format_conversations, batched=True)
    print(f"Dataset: {len(dataset)} examples")

    # Train
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=None,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_steps=WARMUP_STEPS,
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=f"outputs/{task_name}",
            report_to="none",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n",
    )

    stats = trainer.train()
    loss = stats.training_loss
    print(f"\nTraining done — loss: {loss:.4f}")

    # Quick inference test
    print(f"\n--- Inference test ---")
    sys_prompt = TASK_SYSTEM_PROMPTS[task]
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": test_prompts[task_name]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    _ = model.generate(
        **inputs, max_new_tokens=256,
        temperature=0.1, top_p=0.95,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    # Save LoRA adapter
    lora_dir = f"outputs/{task_name}/lora"
    model.save_pretrained(lora_dir)
    tokenizer.save_pretrained(lora_dir)
    print(f"\nLoRA saved to {lora_dir}")

    # Push to HuggingFace
    if HF_TOKEN:
        repo = f"{HF_USERNAME}/grid-gemma4-{task_name}-lora"
        print(f"Pushing to {repo}...")
        model.push_to_hub(repo, token=HF_TOKEN)
        tokenizer.push_to_hub(repo, token=HF_TOKEN)
        print(f"Pushed: https://huggingface.co/{repo}")
    else:
        print("HF_TOKEN not set — skipping push")

    results[task_name] = {"loss": loss, "examples": len(dataset)}

    # Cleanup for next task
    del trainer, dataset, stats
    gc.collect()
    torch.cuda.empty_cache()

    # Unload LoRA for next task
    try:
        model = model.merge_and_unload()
    except Exception:
        pass  # Some versions handle this differently

print(f"\n\n{'='*60}")
print("ALL TASKS COMPLETE")
print(f"{'='*60}")
for t, r in results.items():
    print(f"  {t}: loss={r['loss']:.4f}, examples={r['examples']}")


  TASK: signal_classifier

Unsloth: Making `model.base_model.model.model.language_model` require gradients


Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Dataset: 24 examples


Map (num_proc=6):   0%|          | 0/24 [00:00<?, ? examples/s]

...

Filter (num_proc=6):   0%|          | 0/24 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


Step,Training Loss
1,13.069133
2,13.323791
3,13.386261
4,13.241301
5,13.255633
6,13.280035
7,13.291164
8,13.283914
9,12.542053



Training done — loss: 13.1859

--- Inference test ---


TypeError: string indices must be integers, not 'str'

In [10]:
  # Save signal_classifier that already trained, then continue remaining tasks
  lora_dir = "outputs/signal_classifier/lora"
  model.save_pretrained(lora_dir)
  tokenizer.save_pretrained(lora_dir)
  print(f"signal_classifier LoRA saved to {lora_dir}")

  if HF_TOKEN:
      repo = f"{HF_USERNAME}/grid-gemma4-signal_classifier-lora"
      model.push_to_hub(repo, token=HF_TOKEN)
      tokenizer.push_to_hub(repo, token=HF_TOKEN)
      print(f"Pushed: https://huggingface.co/{repo}")

  results["signal_classifier"] = {"loss": 13.1859, "examples": 0}

  # Cleanup and continue
  del trainer, dataset, stats
  gc.collect()
  torch.cuda.empty_cache()
  try:
      model = model.merge_and_unload()
  except Exception:
      pass

  # Continue with remaining tasks
  remaining = ["anomaly_narrator", "edgar_extractor", "knowledge_mapper"]
  for task_name in remaining:
      print(f"\n{'='*60}")
      print(f"  TASK: {task_name}")
      print(f"{'='*60}\n")

      model = FastModel.get_peft_model(
          model, finetune_vision_layers=False, finetune_language_layers=True,
          finetune_attention_modules=True, finetune_mlp_modules=True,
          r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=3407,
      )

      task = TaskType(task_name)
      dataset = load_dataset_for_training(task)
      dataset = standardize_data_formats(dataset)

      def format_conversations(examples):
          convos = examples["conversations"]
          texts = [tokenizer.apply_chat_template(convo, tokenize=False,
  add_generation_prompt=False).removeprefix("<bos>") for convo in convos]
          return {"text": texts}

      dataset = dataset.map(format_conversations, batched=True)
      print(f"Dataset: {len(dataset)} examples")

      trainer = SFTTrainer(
          model=model, tokenizer=tokenizer, train_dataset=dataset, eval_dataset=None,
          args=SFTConfig(
              dataset_text_field="text", per_device_train_batch_size=BATCH_SIZE,
              gradient_accumulation_steps=GRAD_ACCUM, warmup_steps=WARMUP_STEPS,
              num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, logging_steps=1,
              optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
              seed=3407, output_dir=f"outputs/{task_name}", report_to="none",
              fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
          ),
      )
      trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n",
  response_part="<start_of_turn>model\n")

      stats = trainer.train()
      loss = stats.training_loss
      print(f"\nTraining done — loss: {loss:.4f}")

      lora_dir = f"outputs/{task_name}/lora"
      model.save_pretrained(lora_dir)
      tokenizer.save_pretrained(lora_dir)
      print(f"LoRA saved to {lora_dir}")

      if HF_TOKEN:
          repo = f"{HF_USERNAME}/grid-gemma4-{task_name}-lora"
          model.push_to_hub(repo, token=HF_TOKEN)
          tokenizer.push_to_hub(repo, token=HF_TOKEN)
          print(f"Pushed: https://huggingface.co/{repo}")

      results[task_name] = {"loss": loss, "examples": len(dataset)}
      del trainer, dataset, stats
      gc.collect()
      torch.cuda.empty_cache()
      try:
          model = model.merge_and_unload()
      except Exception:
          pass

  print(f"\n\nALL TASKS COMPLETE")
  for t, r in results.items():
      print(f"  {t}: loss={r['loss']:.4f}")

signal_classifier LoRA saved to outputs/signal_classifier/lora


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(



  TASK: anomaly_narrator



/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Unsloth: Making `model.base_model.model.model.language_model` require gradients


...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,13.928750
2,13.929942
3,13.928998



Training done — loss: 13.9292
LoRA saved to outputs/knowledge_mapper/lora


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(




ALL TASKS COMPLETE
  signal_classifier: loss=13.1859
  anomaly_narrator: loss=13.1275
  edgar_extractor: loss=11.5515
  knowledge_mapper: loss=13.9292


In [11]:
  !zip -r /content/grid_lora_adapters.zip outputs/*/lora/
  from google.colab import files
  files.download('/content/grid_lora_adapters.zip')


  adding: outputs/anomaly_narrator/lora/ (stored 0%)
  adding: outputs/anomaly_narrator/lora/adapter_config.json (deflated 60%)
  adding: outputs/anomaly_narrator/lora/adapter_model.safetensors (deflated 60%)
  adding: outputs/anomaly_narrator/lora/tokenizer.json (deflated 83%)
  adding: outputs/anomaly_narrator/lora/README.md (deflated 65%)
  adding: outputs/anomaly_narrator/lora/chat_template.jinja (deflated 70%)
  adding: outputs/anomaly_narrator/lora/processor_config.json (deflated 69%)
  adding: outputs/anomaly_narrator/lora/tokenizer_config.json (deflated 73%)
  adding: outputs/edgar_extractor/lora/ (stored 0%)
  adding: outputs/edgar_extractor/lora/adapter_config.json (deflated 60%)
  adding: outputs/edgar_extractor/lora/adapter_model.safetensors (deflated 60%)
  adding: outputs/edgar_extractor/lora/tokenizer.json (deflated 83%)
  adding: outputs/edgar_extractor/lora/README.md (deflated 65%)
  adding: outputs/edgar_extractor/lora/chat_template.jinja (deflated 70%)
  adding: outp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>